# UNZA CSC4792 — Kawambwa Town Council Data Pipeline
### End-to-End Local Government Data Acquisition, Extraction, Deduplication & Structuring

**Course:** UNZA CSC4792 — Data Mining & Warehousing / Data Engineering  
**Group:** Team #35  
**Target Authority:** [Kawambwa Town Council](https://www.kawambwacouncil.gov.zm) (Luapula Province, Zambia)  
**Constituencies:** Kawambwa & Pambashe (22 Administrative Wards)  
**Output Specification:** Pipe-delimited (`|`) UTF-8 relational datasets (`db-unza26-csc4792-[DESCRIPTION].csv`)

---

## 1. Project Background & Executive Overview

Kawambwa Town Council is a statutory local government authority in Luapula Province, Zambia, responsible for municipal service delivery, local infrastructure, land administration, and fiscal management across two parliamentary constituencies: **Kawambwa** and **Pambashe**. 

A core objective of this project is transforming unstructured and semi-structured public records—including HTML web portal pages, native PDF tables, multi-year developmental budgets, and scanned public consultation minutes—into standardized, machine-readable datasets for data mining, expenditure tracking, and civic accountability.

### 5 Statutory Governance Domains Covered:
1. **Constituency Development Fund (CDF) Projects (2022–2025):** Complete project tracking across approved, deferred, and updated records with canonical ward normalization and conflict tracking.
2. **Output-Based Budget (OBB) & Financial Statements (2022–2028):** Multi-year expenditure ceilings by economic classification and audited budget vs. actuals under Cash Basis IPSAS.
3. **Integrated Development Plan (IDP 2024–2028):** 10-year Capital Investment Programme (CIP) infrastructure commitments.
4. **Environmental & Social Management Plan (ESMP):** Statutory mitigation matrix for the modern bus station capital project.
5. **Council Resolutions & Consultations (2024–2025):** Ordinary Council meeting decisions and ratepayer public consultation assemblies.

## 2. Environment Setup & Core Dependencies

The pipeline relies on:
- `requests` & `urllib3` for SSL-bypassing web scraping and PDF acquisition.
- `beautifulsoup4` for HTML portal traversal.
- `pdfplumber` for geometric table and cell boundary extraction.
- `pypdfium2` & Apple Vision framework for optical character recognition (OCR) of scanned minutes.
- `pandas` for dataframe representation, type casting, and validation.

In [1]:
import os
import re
import csv
import json
import warnings
import urllib3
import requests
import pandas as pd
import pdfplumber
from collections import defaultdict
from difflib import SequenceMatcher

# Suppress SSL and parser warnings
warnings.filterwarnings('ignore')
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 1000)

print("Environment configured successfully. Pandas version:", pd.__version__)

/Users/frankmeyo/Downloads/unza26-csc4792-kawambwa-council/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Environment configured successfully. Pandas version: 2.3.3


## 3. Web Crawling, PDF Discovery & SSL Workaround

### ⚠️ Methodology Note: The SSL / `verify=False` Workaround
The official Kawambwa Town Council web portal (`kawambwacouncil.gov.zm`) operates with a self-signed or misconfigured SSL/TLS certificate chain. Standard HTTPS requests in Python fail immediately with `SSLCertVerificationError`. 

To ensure continuous data acquisition without manual browser intervention:
1. All HTTP request sessions configure `verify=False`.
2. `urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)` suppresses stdout flood.
3. A polite crawl rate (0.5s delay) was maintained to prevent overloading local municipal servers.

The crawl identified **26 HTML pages** and cataloged **73 statutory PDF publications**.

In [2]:
def summarize_discovered_documents():
    pdf_dir = "pdf_downloads"
    if os.path.exists(pdf_dir):
        files = [f for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]
        print(f"Total statutory PDFs downloaded and stored in '{pdf_dir}': {len(files)}")
        print("Sample PDF filenames:")
        for f in sorted(files)[:8]:
            print(f"  - {f}")
    else:
        print(f"Directory {pdf_dir} not found. Ensure scraping phase is complete.")

summarize_discovered_documents()

Total statutory PDFs downloaded and stored in 'pdf_downloads': 73
Sample PDF filenames:
  - 2022-PAMBASHE-LOANS-APLLICATION-LIST.pdf
  - 2024-Annual-Perfomance-Report.pdf
  - 2025-KAWAMBWA-COMM-PROJECTS-UPDATED.pdf
  - 2025-Pambashe-CommProjects-updated.pdf
  - APPLICATION-FORM-FOR-COMMUNITY-PROJECTS.pdf
  - APPROVED-KAWAMBWA-COMMUNITY-PROJECTS-2025.pdf
  - APPROVED-PAMBASHE-COMMUNITY-PROJECTS-2025.pdf
  - Act-No.-3-The-Data-Protection-Act-2021_0-2.pdf


## 4. Statutory Document Categorization

Using regex heuristics on document nomenclature and metadata in `04_categorize_and_extract.py`, all 73 publications were mapped into distinct statutory domains to establish extraction pipelines.

In [3]:
CATEGORIES = {
    "cdf_and_community_projects": [
        r"COMMUNITY-PROJECTS", r"CommProjects", r"APPLICATION-FORM-FOR-COMMUNITY",
        r"SECONDARY-BORADING", r"SKILLS-DEVELOPMENT", r"FORM-FOR-YOUTH-WOMEN",
        r"Grant-Application-Form", r"Constituency-Development-Fund"
    ],
    "budgets_and_financial_statements": [
        r"BUDGET", r"Financial-Statements", r"FINANCIAL-STATEMENTS", r"AUDIT",
        r"PROPERTY-RATES", r"LA-Financial-Regulations", r"Public-Finance-Management",
        r"Public-Procurement", r"Perfomance-Report"
    ],
    "idp_and_strategic_plans": [
        r"IDP", r"NATIONAL-DEVELOPMENT-PLAN"
    ],
    "council_meeting_minutes": [
        r"MINUTES", r"STAKEHOLDER", r"tenants-meeting"
    ],
    "loans_and_empowerment": [
        r"LOAN", r"loans"
    ],
    "acts_policies_and_charters": [
        r"Charter", r"Act-No", r"Act-12", r"Act-20", r"Act\b", r"Acts",
        r"Procedures_ZDSP", r"Commitment-Plan", r"ESMP", r"SI-", r"si_"
    ],
    "newsletters_and_press": [
        r"Newsletter", r"Newletter", r"Press-statement", r"Advert"
    ]
}

def categorize_filename(filename: str) -> str:
    for cat_name, patterns in CATEGORIES.items():
        for pat in patterns:
            if re.search(pat, filename, re.IGNORECASE):
                return cat_name
    return "other_documents"

# Display category distribution for downloaded files
if os.path.exists("pdf_downloads"):
    files = [f for f in os.listdir("pdf_downloads") if f.lower().endswith(".pdf")]
    cat_counts = defaultdict(int)
    for f in files:
        cat_counts[categorize_filename(f)] += 1
    
    df_cats = pd.DataFrame(list(cat_counts.items()), columns=["Category", "PDF Count"]).sort_values("PDF Count", ascending=False)
    display(df_cats)

,Category,PDF Count
0,acts_policies_and_charters,20
3,cdf_and_community_projects,16
2,budgets_and_financial_statements,12
6,council_meeting_minutes,8
1,loans_and_empowerment,6
4,newsletters_and_press,4
7,other_documents,4
5,idp_and_strategic_plans,3


## 5. Dataset 1: Constituency Development Fund (CDF) Projects (2022–2025)

The Constituency Development Fund is Zambia's flagship decentralized financing mechanism. In Kawambwa, CDF records are published as annual PDF tables across Kawambwa and Pambashe constituencies.

### ⚠️ Methodology Decisions & Data Engineering Insights:

1. **The `[:40]` Truncation Bug:**  
   Early deduplication prototypes truncated project titles to 40 characters for fast fuzzy hashing (`title[:40]`). This introduced severe false collisions: distinct projects like `"Construction of 1x3 Classroom Block at Mweo Primary"` and `"Construction of 1x3 Classroom Block at Chibote Secondary"` collapsed together. We eliminated slicing, implemented complete canonical normalization (`norm_for_match`), and applied `SequenceMatcher` with a strict similarity threshold ($\ge 0.90$).

2. **Exact-Duplicate vs. Amount-Aware Matching:**  
   Naive deduplication matching on `(ward, project_name, year, source_file)` collapsed legitimately distinct budget entries. For example, in 2023, four separate desk procurement entries with different amounts (`K532,000`, `K632,000`, `K632,000`, and `K3,074,000`) collapsed into one row, discarding ~K4M in distinct public allocations. Adding `amount` to the matching key ensured distinct budget lines remained intact, while collapsing genuine blank-amount extraction duplicates (e.g. Mweo CRB and Ilombe Ablution Block) and keeping the version with the highest completeness score.

3. **2025 Cross-File Conflict Resolution (The Mawaya Case Study):**  
   In 2025, projects were published across multiple overlapping PDF files (`APPROVED`, `UPDATED`, and `DEFERRED`). A direct cross-file status conflict arose with the project **"Construction of Health Facility at Mawaya"** located in **Mulunda Ward** (Mawaya is a locality within Mulunda, not a standalone ward). This single project appeared with `status = Approved` in `APPROVED-PAMBASHE-COMMUNITY-PROJECTS-2025.pdf`, but was simultaneously listed with `status = Deferred` in `DEFERRED-2025-Pambashe-CommProjects-updated.pdf`. We resolved this contradiction by implementing an **Authority Hierarchy** (`APPROVED` [Rank 3] > `UPDATED` [Rank 2] > `DEFERRED` [Rank 1]), which retained the authoritative ministerial approval status while explicitly setting `data_conflict = True` to preserve full auditability for downstream researchers.

4. **Ward Canonical Normalization:**  
   Source PDFs had typos and spelling variants across 22 wards (e.g., `Chipili` $\rightarrow$ `Chimpili`, `Pampashe` $\rightarrow$ `Pambashe`, `Ntumacushi` $\rightarrow$ `Ntumbachushi`). All wards were mapped to their canonical Gazette spellings.

In [4]:
# Execute CDF structuring pipeline
import subprocess

cdf_csv = "db-unza26-csc4792-kawambwa_cdf_projects.csv"
if not os.path.exists(cdf_csv):
    subprocess.run(["python3", "05_structure_cdf_projects.py"], check=True)

df_cdf = pd.read_csv(cdf_csv, sep="|")
print(f"Loaded CDF Projects Dataset: {len(df_cdf)} rows, {df_cdf.shape[1]} columns")
conflicts = (df_cdf['data_conflict'].astype(str).str.lower() == 'true').sum()
print(f"Conflicts Flagged: {conflicts} rows")
display(df_cdf.head(10))

Loaded CDF Projects Dataset: 442 rows, 10 columns
Conflicts Flagged: 1 rows


,ward,project_name,year,status,amount,category,project_type,constituency,source_file,data_conflict
0,Kala,Helath Post At Muchulila,2022,Approved,"770,000.00",Health,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
1,Fisaka,construction of a 1*3 Class Room Block and abl...,2022,Approved,"1,328,000.00",Education,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
2,Senga,2 Staff Houses For Health Workers at Luatula H...,2022,Approved,"810,000.00",Health,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
3,Ng'ona,Bus Station Paving,2022,Approved,"575,846.21",Bus Stations And Markets,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
4,Ntumbachushi,"Construction Of 1*3 Crb, And Ablution Block At...",2022,Approved,"1,109,000.21",Education,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
5,Kawambwa,Construction Of An Ablution Block at Munkanta ...,2022,Approved,"680,746.00",Education,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
6,Iyanga,Grading And Gravelling Of Road From Muyembe To...,2022,Approved,NaN,Roads,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
7,All Wards,Procurement Of 300 Desks,2022,Approved,"594,100.00",Education,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
8,Senga,Construction Of 1*3 Classroom Block at Mufwaya...,2022,Approved,"503,000.00",Education,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False
9,Lushiba,"Construction Of Borehole, Staff House at Mwang...",2022,Approved,"675,000.21",Health,NaN,Kawambwa,KAWAMBWA-COMMUNITY-PROJECTS-year2022.pdf,False


In [5]:
# Summary statistics on CDF projects by year and approval status
cdf_summary = df_cdf.groupby(['year', 'status']).size().unstack(fill_value=0)
print("CDF Project Counts by Year & Status:")
display(cdf_summary)

print("Top 5 Wards by Number of CDF Projects:")
display(df_cdf['ward'].value_counts().head(5))

CDF Project Counts by Year & Status:


status,Approved,Deferred / Not Approved
year,,
2022,54,14
2023,30,16
2024,32,146
2025,39,111


Top 5 Wards by Number of CDF Projects:


ward
Ng'ona                               40
Filenge                              38
All Wards                            36
Chikanda                             30
Constituency-wide / Not Specified    27
Name: count, dtype: int64

## 6. Dataset 2 & 3: Output-Based Budget (OBB) & Audited Financial Statements

### Dataset 2: `db-unza26-csc4792-kawambwa_budget_obb.csv` (105 rows)
Extracted from `KTC-OBB-BUDGET-2026-to-2027.pdf`, this dataset captures Kawambwa's Medium-Term Expenditure Framework (MTEF). It includes:
- **Revenues (Pages 3–4):** Local taxes/rates, fees & charges, national equalization grants, and capital grants across 2024–2028.
- **Expenditures by Economic Classification (Pages 4–5):** Personnel emoluments, goods & services, capital expenditure, and transfers (2024–2026).

### Dataset 3: `db-unza26-csc4792-kawambwa_financials_2022.csv` (42 rows)
Extracted from `Kawambwa-Town-Council-Audited-Financial-Statements-2022.pdf`, capturing the statutory **Statement of Comparison of Budget and Actual Amounts** prepared under Cash Basis IPSAS. It contains:
- Receipts (Local taxes, fees, licenses, national equalization grants)
- Payments (Compensation of employees, use of goods and services, social benefits)
- Original Budget, Adjustments, Final Budget, Actual Outturn, Variance, and Performance %.

In [6]:
obb_csv = "db-unza26-csc4792-kawambwa_budget_obb.csv"
fin_csv = "db-unza26-csc4792-kawambwa_financials_2022.csv"

if not os.path.exists(obb_csv) or not os.path.exists(fin_csv):
    subprocess.run(["python3", "06_structure_budget_financials.py"], check=True)

df_obb = pd.read_csv(obb_csv, sep="|")
df_fin = pd.read_csv(fin_csv, sep="|")

print(f"OBB Budget Dataset: {len(df_obb)} rows")
display(df_obb.head(6))

print(f"2022 Audited Financial Statements Dataset: {len(df_fin)} rows")
display(df_fin.head(6))

OBB Budget Dataset: 105 rows


,budget_section,classification_level,code,description,amount_2024,amount_2025,amount_2026,amount_2027,amount_2028,source_file,page
0,Revenue,Detailed Line Item,01,Local taxes/rates,NaN,NaN,NaN,NaN,NaN,KTC-OBB-BUDGET-2026-to-2027.pdf,3
1,Revenue,Detailed Line Item,001,Residential,NaN,NaN,"262,670.00","262,670.00","262,670.00",KTC-OBB-BUDGET-2026-to-2027.pdf,3
2,Revenue,Detailed Line Item,002,Commercial,NaN,NaN,"161,864.00","161,864.00","161,864.00",KTC-OBB-BUDGET-2026-to-2027.pdf,3
3,Revenue,Detailed Line Item,003,Industrial,NaN,NaN,"84,838.00","84,838.00","84,838.00",KTC-OBB-BUDGET-2026-to-2027.pdf,3
4,Revenue,Detailed Line Item,004,Hospitality,NaN,NaN,"93,669.00","93,669.00","93,669.00",KTC-OBB-BUDGET-2026-to-2027.pdf,3
5,Revenue,Detailed Line Item,NaN,SubItem Total,NaN,NaN,NaN,"603,041.00","603,041.00",KTC-OBB-BUDGET-2026-to-2027.pdf,3


2022 Audited Financial Statements Dataset: 42 rows


,statement_name,section,item_name,original_budget,adjustments,final_budget,actual_amount,pct_performance,variance,pct_variance,year,source_file,page
0,Statement of Comparison of Budget and Actual A...,Receipts,Local taxes,"652,945.00",0.00,"652,945.00","468,731.00",72%,"184,214.00",28%,2022,Kawambwa-Town-Council-Audited-Financial-Statem...,12
1,Statement of Comparison of Budget and Actual A...,Receipts,Fees and Charges,"5,095,804.00",0.00,"5,095,804.00","2,471,780.00",49%,"2,624,024.00",51%,2022,Kawambwa-Town-Council-Audited-Financial-Statem...,12
2,Statement of Comparison of Budget and Actual A...,Receipts,Licences,"112,040.00",0.00,"112,040.00","243,628.00",217%,"-131,588.00",-117%,2022,Kawambwa-Town-Council-Audited-Financial-Statem...,12
3,Statement of Comparison of Budget and Actual A...,Receipts,Levies,"26,700.00",0.00,"26,700.00","25,441.00",95%,"1,259.00",5%,2022,Kawambwa-Town-Council-Audited-Financial-Statem...,12
4,Statement of Comparison of Budget and Actual A...,Receipts,Permits,"351,550.00",0.00,"351,550.00","332,168.00",94%,"19,382.00",6%,2022,Kawambwa-Town-Council-Audited-Financial-Statem...,12
5,Statement of Comparison of Budget and Actual A...,Receipts,Local Government Equalisation Fund,"9,321,973.00","1,490,500.00","10,812,473.00","10,152,215.00",94%,"660,258.00",6%,2022,Kawambwa-Town-Council-Audited-Financial-Statem...,12


## 7. Dataset 4 & 5: Integrated Development Plan (IDP) & Bus Station ESMP

### Dataset 4: `db-unza26-csc4792-kawambwa_idp_projects.csv` (43 rows)
Extracted from `Kawambwa-IDP.pdf` (pages 154–157), this captures the 10-year **Capital Investment Programme (CIP)** across agriculture, civic infrastructure, water/sanitation, and transportation with 2024–2028 multi-year cost allocations.

### Dataset 5: `db-unza26-csc4792-kawambwa_esmp_bus_station.csv` (14 rows)
Extracted from `KTC-ESMP-MODERN-BUS-STATION.pdf`, capturing the Environmental & Social Management Plan mitigation matrix for Kawambwa's modern bus station.

### ⚠️ Methodology Decision: The Continuation Line Wrap-Around Fix
In `KTC-ESMP-MODERN-BUS-STATION.pdf`, table cells describing potential impacts and mitigation measures wrapped across multiple lines in the PDF stream. Naive table extractors created orphan rows before items 5 and 8 with empty `impact_no` and disconnected text fragments. We applied single-non-empty-cell lookback merging to absorb wrapped lines back into parent items 4 and 7 before emitting clean rows.

In [7]:
idp_csv = "db-unza26-csc4792-kawambwa_idp_projects.csv"
esmp_csv = "db-unza26-csc4792-kawambwa_esmp_bus_station.csv"

if not os.path.exists(idp_csv) or not os.path.exists(esmp_csv):
    subprocess.run(["python3", "07_structure_idp_docs.py"], check=True)

df_idp = pd.read_csv(idp_csv, sep="|")
df_esmp = pd.read_csv(esmp_csv, sep="|")

print(f"IDP CIP Projects Dataset: {len(df_idp)} rows")
display(df_idp.head(5))

print(f"Modern Bus Station ESMP Mitigation Dataset: {len(df_esmp)} rows")
display(df_esmp.head(5))

IDP CIP Projects Dataset: 43 rows


,programme,project_name,location_priority,cost_2024,cost_2025,cost_2026,cost_2027,cost_2028,responsible_agency,source_file,page
0,Capital Infrastructure,Construction of agricultural and livestock res...,Pambashe and Kawambwa,0.00,"500,000.00","500,000.00",0.00,NaN,"LA,MOA, MOL",Kawambwa-IDP.pdf,154
1,Livestock development,Construction of Deep Tanks,District-wide,"180,000.00","180,000.00","180,000.00","180,000.00","180,000.00",MOFL,Kawambwa-IDP.pdf,154
2,Capital Infrastructure,Construction of Harbotour,Ngona,"500,000.00",0.00,0.00,0.00,0.00,"LA, MOFL",Kawambwa-IDP.pdf,154
3,Capital Infrastructure,Construction of Slaughter Slab,Ngona,"150,000.00",0.00,0.00,0.00,0.00,"LA, MOFL",Kawambwa-IDP.pdf,154
4,Early warning and surveillance systems,Construction of of Early Warning Office,Pambashe,0.00,0.00,"450,000.00",0.00,0.00,"LA,MET,MOA",Kawambwa-IDP.pdf,154


Modern Bus Station ESMP Mitigation Dataset: 14 rows


,impact_no,impact_category,potential_impact,mitigation_measure,responsibility,timeline,source_file,page
0,1,Loss of flora / fauna,During site Preparation,Restrict the activities within the designated ...,Contractor/Local Authority,Short Term,KTC-ESMP-MODERN-BUS-STATION.pdf,8
1,2,Dust pollution,Dust from site preparation and construction wo...,Employ dust suppression mechanisms such as com...,Contractor/Local Authority,Short Term,KTC-ESMP-MODERN-BUS-STATION.pdf,8
2,3,Contamination of the environment through leakages,Degradation of soil and contamination of surfa...,Constant checks of the vehicles and equipment ...,Contractor/Local Authority,Short Term,KTC-ESMP-MODERN-BUS-STATION.pdf,8
3,4,Safety and Risk of workers,Loss of life,Ensure that all workers are briefed on potenti...,Contractor/Local Authority,Short Term,KTC-ESMP-MODERN-BUS-STATION.pdf,8
4,5,Generation of sewage,Contaminating surface water and underground water,All sewage will be directed / connected to the...,Contractor/Local Authority,Long term,KTC-ESMP-MODERN-BUS-STATION.pdf,9


## 8. Dataset 6: Council Resolutions & Administrative Records (2024–2025)

### Dataset 6: `db-unza26-csc4792-kawambwa_council_resolutions.csv` (44 rows)
Stitching together statutory resolutions from:
1. **Ordinary Council Minutes (2024–2025):** 1st Ordinary 2024, 2nd Ordinary 2024, 1st Ordinary 2025, and 2nd Ordinary 2025.
2. **Consultative Minutes:** Ratepayer property rates review assembly (via Apple Vision OCR) and modern bus station trader engagements.

### ⚠️ Methodology Decision: Meeting Sequence Tracking
Both `MINUTES-OF-FIRST-ORDINARY-COUNCIL-2025.pdf` (held 5 June 2025) and `MINUTES-OF-SECOND-ORDINARY-COUNCIL-2025.pdf` (held 5 August 2025) reuse the minute numbering template `KTC/01/06/2025` through `KTC/09/06/2025`. To prevent primary key collisions, the pipeline introduces a `meeting_sequence` attribute (e.g. `1st Ordinary 2025`, `2nd Ordinary 2025`) ensuring every statutory resolution is uniquely addressable.

In [8]:
res_csv = "db-unza26-csc4792-kawambwa_council_resolutions.csv"

if not os.path.exists(res_csv):
    subprocess.run(["python3", "08_structure_admin_minutes.py"], check=True)

df_res = pd.read_csv(res_csv, sep="|")
print(f"Council Resolutions Dataset: {len(df_res)} rows")
display(df_res.head(8))

print("Resolutions Count by Meeting Sequence:")
display(df_res['meeting_sequence'].value_counts())

Council Resolutions Dataset: 44 rows


,meeting_sequence,meeting_date,minute_no,subject,decision_type,resolution_text,proposer_seconder,source_file
0,1st Ordinary 2024,2024-02-27,DCM/39/02/2024,DECLARATION OF INTEREST,Declaration of Interest,His Worship Kalumba Chifumbe declared interest...,NaN,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
1,1st Ordinary 2024,2024-02-27,KTC/40/02/2024,CHAIRMAN’S COMMUNICATION,Council Information / Deliberation,His Worship the Council Chairperson welcomed a...,NaN,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
2,1st Ordinary 2024,2024-02-27,KTC/41/02/2024,ELECTION OF DEPUTY COUNCIL CHAIRPERSON.,Statutory Election,Elected Mulenga Daniel C. as Deputy Council Ch...,NaN,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
3,1st Ordinary 2024,2024-02-27,KTC/42/02/2024,CONFIRMATION OF PREVIOUS MINUTES OF THE FOURTH...,Confirmation of Minutes,The Minutes of the FOURTH ORDINARY COUNCIL MEE...,Cllr. Mulenga Daniel / Cllr. Chansa Fanwell,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
4,1st Ordinary 2024,2024-02-27,KTC/43/02/2024,MATTERS ARISING,Matters Arising / Directives,ITEM 1: VIDE MINUTE NUMBER HESS/206/11/2023: U...,NaN,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
5,1st Ordinary 2024,2024-02-27,KTC/44/02/2024,ADOPTION OF THE MINUTES OF THE AUDIT COMMITTEE...,Committee Adoption,1. The Minutes of the Audit committee meeting ...,NaN,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
6,1st Ordinary 2024,2024-02-27,KTC/45/02/2024,ADOPTION OF THE MINUTES OF THE HEALTH ENVIRONM...,Committee Adoption,"1. Minutes of the Health, Environment and Soci...",NaN,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf
7,1st Ordinary 2024,2024-02-27,KTC/46/02/2024,"ADPTION OF MINUTES OF THE PLANS, WORKS, REAL E...",Committee Adoption,"1.) The Minutes of the Plans, Works, Developme...",the committee chairperson Councilor Mubanga Ke...,MINUTES-FIRST-ORDINARY-27TH-FEB-2024.pdf


Resolutions Count by Meeting Sequence:


meeting_sequence
1st Ordinary 2024                            11
2nd Ordinary 2024                            10
1st Ordinary 2025                             8
2nd Ordinary 2025                             8
Stakeholder Engagement Modern Bus Station     4
Stakeholder Consultation 2025 Budget          3
Name: count, dtype: int64

## 9. Final Data Integrity & Richness Sanity-Check Summary

Here we verify all 6 structured CSVs, checking file size, line counts, non-empty attributes, and coverage against the UNZA CSC4792 specification.

In [9]:
datasets = [
    ("CDF Community Projects (2022-2025)", "db-unza26-csc4792-kawambwa_cdf_projects.csv"),
    ("Output-Based Budget MTEF (2024-2028)", "db-unza26-csc4792-kawambwa_budget_obb.csv"),
    ("Audited Financial Statements (2022)", "db-unza26-csc4792-kawambwa_financials_2022.csv"),
    ("IDP Capital Investment Programme", "db-unza26-csc4792-kawambwa_idp_projects.csv"),
    ("Bus Station ESMP Mitigation Matrix", "db-unza26-csc4792-kawambwa_esmp_bus_station.csv"),
    ("Council Resolutions & Consultations", "db-unza26-csc4792-kawambwa_council_resolutions.csv"),
]

summary_rows = []
for desc, fname in datasets:
    if os.path.exists(fname):
        df = pd.read_csv(fname, sep="|")
        size_kb = os.path.getsize(fname) / 1024
        summary_rows.append({
            "Description": desc,
            "File Name": fname,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Size (KB)": round(size_kb, 1),
            "Column List": ", ".join(df.columns[:4]) + ("..." if len(df.columns) > 4 else "")
        })
    else:
        summary_rows.append({
            "Description": desc,
            "File Name": fname,
            "Rows": 0,
            "Columns": 0,
            "Size (KB)": 0,
            "Column List": "MISSING"
        })

df_summary = pd.DataFrame(summary_rows)
print("=" * 80)
print("UNZA CSC4792 - KAWAMBWA TOWN COUNCIL STRUCTURED DATASETS AUDIT")
print("=" * 80)
display(df_summary)
print(f"\nTotal Structured Records Across All Domains: {df_summary['Rows'].sum()} rows")

UNZA CSC4792 - KAWAMBWA TOWN COUNCIL STRUCTURED DATASETS AUDIT


,Description,File Name,Rows,Columns,Size (KB),Column List
0,CDF Community Projects (2022-2025),db-unza26-csc4792-kawambwa_cdf_projects.csv,442,10,64.9,"ward, project_name, year, status..."
1,Output-Based Budget MTEF (2024-2028),db-unza26-csc4792-kawambwa_budget_obb.csv,105,11,12.6,"budget_section, classification_level, code, de..."
2,Audited Financial Statements (2022),db-unza26-csc4792-kawambwa_financials_2022.csv,42,13,8.3,"statement_name, section, item_name, original_b..."
3,IDP Capital Investment Programme,db-unza26-csc4792-kawambwa_idp_projects.csv,43,11,6.9,"programme, project_name, location_priority, co..."
4,Bus Station ESMP Mitigation Matrix,db-unza26-csc4792-kawambwa_esmp_bus_station.csv,14,8,3.2,"impact_no, impact_category, potential_impact, ..."
5,Council Resolutions & Consultations,db-unza26-csc4792-kawambwa_council_resolutions...,44,8,16.3,"meeting_sequence, meeting_date, minute_no, sub..."



Total Structured Records Across All Domains: 690 rows
